# **RIDGE regression – Stock-wise analysis**

In this notebook we aim to optimize regularization parameter $\lambda$ for every single stock ticker included in database. Using this we can optimize the overall performance more however, needs to be strongly checked on testing as we are balancing on the edge of overfit, of course.

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from DataFramePrep import generate_TrainingDataFrame

# Metrics to measure successfulness
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler

# ML Stuff
from sklearn.linear_model import Ridge

# In case of convergence problem, supress warning
import warnings

#warnings.filterwarnings("ignore")

In [ ]:
# Pre-processed data preparation
TrainingDataFrame, tickers, historic_columns, StockDataDatabase = generate_TrainingDataFrame()

# **RIDGE regression – per stock**

In [ ]:
# To track how well predictions work, all metrics -> dominance, MAPE/APE and benchmark performance
performance_tracker_regularization = {}

for ticker in tickers["Ticker"]:
    performance_tracker_regularization[ticker] = []
    for Alpha in range(120, 261, 10):
        # Bounded training-validation set
        Stock_Data = pd.read_sql(
            f"SELECT * FROM StockData WHERE Ticker='{ticker}' AND Date>='2017-09-14' AND Date<='2022-12-31'",
            con=StockDataDatabase, parse_dates=["Date"]
        )
            
        Stock_Data = pd.merge(Stock_Data, TrainingDataFrame, on="Date")
        Stock_Data["Target"] = Stock_Data["Close"]
        Stock_Data = Stock_Data.dropna().reset_index(drop=True)

        training_length = 4
        prediction_length = 1

        MAPEs = []

        for window_start in range(len(Stock_Data) - (training_length + prediction_length)):
            # Define feature and target windows
            Training_Features = Stock_Data[historic_columns].iloc[window_start:window_start+training_length]
            Training_Target = Stock_Data["Target"].iloc[window_start:window_start+training_length]

            Test_Features = Stock_Data[historic_columns].iloc[
                window_start+training_length : window_start+training_length+prediction_length
            ]
            Test_Target = Stock_Data["Target"].iloc[
                window_start+training_length : window_start+training_length+prediction_length
            ]

            # Skip empty slices
            if Training_Features.empty or Test_Features.empty:
                continue
            
            # Scale only selected features
            scaler = StandardScaler()
            Training_Features = scaler.fit_transform(Training_Features)
            Test_Features = scaler.transform(Test_Features)

            # Fit KNN
            MODEL = Ridge(alpha=Alpha)
            MODEL.fit(Training_Features, Training_Target)

            # Predict and evaluate
            prediction = MODEL.predict(Test_Features)
            MAPEs.append(100 * mean_absolute_percentage_error(Test_Target, prediction))
            
        # Save results
        performance_tracker_regularization[ticker].append((Alpha, np.mean(MAPEs)))
        print(Alpha, np.mean(MAPEs))


In [ ]:
# Optimal processing
parameters = performance_tracker_regularization.copy()
optimal = {}

for ticker in tickers["Ticker"]:
    best_ones = sorted(parameters[ticker], key=lambda x: x[1])
    optimal[ticker] = best_ones[0][0]
    print(ticker, best_ones[0][0])

ACN 250
ADBE 220
AMD 170
AKAM 140
APH 210
ADI 260
AAPL 260
AMAT 260
ANET 190
ADSK 190
AVGO 240
CDNS 240
CDW 230
CSCO 260
CTSH 260
GLW 240
DELL 140
ENPH 170
EPAM 210
FFIV 180
FICO 250
FSLR 150
FTNT 190
IT 170
GEN 180
GDDY 190
HPE 170
HPQ 210
IBM 160
INTC 250
INTU 190
JBL 170
KEYS 260
KLAC 260
LRCX 250
MCHP 260
MU 220
MSFT 260
MPWR 260
MSI 260
NTAP 190
NVDA 200
NXPI 250
ON 220
ORCL 170
PANW 200
PTC 180
QCOM 260
ROP 230
CRM 170
STX 170
NOW 210
SWKS 260
SMCI 170
SNPS 220
TEL 220
TDY 200
TER 220
TXN 260
TRMB 160
TYL 190
VRSN 250
WDC 170
WDAY 160
ZBRA 210
ACN 250
ADBE 220
AMD 170
AKAM 140
APH 210
ADI 260
AAPL 260
AMAT 260
ANET 190
ADSK 190
AVGO 240
CDNS 240
CDW 230
CSCO 260
CTSH 260
GLW 240
DELL 140
ENPH 170
EPAM 210
FFIV 180
FICO 250
FSLR 150
FTNT 190
IT 170
GEN 180
GDDY 190
HPE 170
HPQ 210
IBM 160
INTC 250
INTU 190
JBL 170
KEYS 260
KLAC 260
LRCX 250
MCHP 260
MU 220
MSFT 260
MPWR 260
MSI 260
NTAP 190
NVDA 200
NXPI 250
ON 220
ORCL 170
PANW 200
PTC 180
QCOM 260
ROP 230
CRM 170
STX 170
NOW 210

In [13]:
for key, value in optimal.items():
    print(f"{key} {value}")

ACN 250
ADBE 220
AMD 170
AKAM 140
APH 210
ADI 260
AAPL 260
AMAT 260
ANET 190
ADSK 190
AVGO 240
CDNS 240
CDW 230
CSCO 260
CTSH 260
GLW 240
DELL 140
ENPH 170
EPAM 210
FFIV 180
FICO 250
FSLR 150
FTNT 190
IT 170
GEN 180
GDDY 190
HPE 170
HPQ 210
IBM 160
INTC 250
INTU 190
JBL 170
KEYS 260
KLAC 260
LRCX 250
MCHP 260
MU 220
MSFT 260
MPWR 260
MSI 260
NTAP 190
NVDA 200
NXPI 250
ON 220
ORCL 170
PANW 200
PTC 180
QCOM 260
ROP 230
CRM 170
STX 170
NOW 210
SWKS 260
SMCI 170
SNPS 220
TEL 220
TDY 200
TER 220
TXN 260
TRMB 160
TYL 190
VRSN 250
WDC 170
WDAY 160
ZBRA 210
